# 📖 Notebook 2: Cache Coherence & Invalidation

In Notebook 1 we learned how to **partition** data across cache nodes. But what if a node
crashes? What if we want replicas for availability? Now we have **multiple copies** of the
same data, and they can get out of sync.

This notebook covers:
- **Replication**: copying data to multiple nodes for fault tolerance
- **Coherence**: keeping copies in sync when data changes
- **Invalidation**: removing stale data across a multi-node cluster

## Learning Objectives

By the end of this notebook, you'll understand:
- Why replication is needed and the trade-offs between sync/async replication
- Write-invalidate vs write-update strategies
- TTL-based invalidation and its limitations
- How to detect and measure staleness across replicas
- Pub/Sub-based invalidation for real-time consistency

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/distributed-cache
docker compose up -d
```

### RedisInsight (Redis GUI)
- **URL**: http://localhost:5540
- Add each node: `localhost:6381`, `localhost:6382`, `localhost:6383`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import redis
import time
import threading
import json

# Our 3 cache nodes
NODES = {
    "node-1": redis.Redis(host="localhost", port=6381, decode_responses=True),
    "node-2": redis.Redis(host="localhost", port=6382, decode_responses=True),
    "node-3": redis.Redis(host="localhost", port=6383, decode_responses=True),
}

# Flush all nodes for a clean start
for name, client in NODES.items():
    client.flushall()
    try:
        client.ping()
        print(f"✅ {name} is up")
    except Exception as e:
        print(f"❌ {name} failed: {e}")

## 🔄 Part 1: Why Replicate?

With partitioning, each key lives on **one** node. If that node crashes, the data is gone.

```
Before crash:   node-1: {A, B}    node-2: {C, D}    node-3: {E, F}
Node-2 crashes: node-1: {A, B}    node-2: 💀         node-3: {E, F}
Result:         Keys C and D are LOST → cache misses → database gets hammered
```

**Replication** keeps copies on multiple nodes so data survives failures.

The challenge: how do you keep replicas in sync when data changes?

In [ ]:
# Let's build a simple replicated cache and see the coherence problem

class ReplicatedCache:
    """
    A cache that writes to a primary node and replicates to backup nodes.
    This shows the basic idea of replication — we'll explore different
    replication strategies below.
    """

    def __init__(self, nodes: dict, replication_factor: int = 2):
        self.nodes = nodes
        self.node_names = list(nodes.keys())
        # replication_factor=2 means data lives on 2 nodes (primary + 1 replica)
        self.replication_factor = min(replication_factor, len(nodes))

    def _get_replicas(self, key: str) -> list[str]:
        """Pick which nodes should hold this key (primary + replicas)."""
        import hashlib
        h = int(hashlib.md5(key.encode()).hexdigest(), 16)
        primary_idx = h % len(self.node_names)
        replicas = []
        for i in range(self.replication_factor):
            idx = (primary_idx + i) % len(self.node_names)
            replicas.append(self.node_names[idx])
        return replicas

    def get(self, key: str) -> str | None:
        """Read from the primary node."""
        replicas = self._get_replicas(key)
        primary = replicas[0]
        return self.nodes[primary].get(key)

    def get_from_any(self, key: str) -> dict:
        """Read from ALL replicas — useful for checking coherence."""
        replicas = self._get_replicas(key)
        results = {}
        for node_name in replicas:
            results[node_name] = self.nodes[node_name].get(key)
        return results


cache = ReplicatedCache(NODES, replication_factor=2)

# Show which nodes each key is assigned to
print("🗂️  Replica assignment (replication factor = 2)")
print("=" * 50)
for key in ["user:alice", "user:bob", "user:charlie"]:
    replicas = cache._get_replicas(key)
    print(f"  {key:<16} → primary: {replicas[0]}, replica: {replicas[1]}")

---

## 🔄 Part 2: Synchronous vs Asynchronous Replication

### Synchronous Replication
Write to **all replicas before** confirming to the client.  
✅ Strong consistency — all replicas always agree  
❌ Slower writes — must wait for the slowest replica  
❌ Lower availability — if any replica is down, writes fail  

### Asynchronous Replication  
Write to the **primary only**, replicate in the background.  
✅ Fast writes — only wait for primary  
✅ Higher availability — writes succeed even if replicas are down  
❌ Eventual consistency — replicas may serve stale data briefly

In [ ]:
# Synchronous Replication: write to ALL replicas before returning

import statistics


def sync_write(cache: ReplicatedCache, key: str, value: str) -> float:
    """Write to all replicas synchronously. Returns time taken."""
    start = time.time()
    replicas = cache._get_replicas(key)
    for node_name in replicas:
        cache.nodes[node_name].set(key, value)
    elapsed = (time.time() - start) * 1000
    return elapsed


# Asynchronous Replication: write to primary, replicate later

def async_write(cache: ReplicatedCache, key: str, value: str, delay_ms: int = 100) -> float:
    """Write to primary immediately, replicate to others after a delay."""
    start = time.time()
    replicas = cache._get_replicas(key)

    # Write to primary immediately
    primary = replicas[0]
    cache.nodes[primary].set(key, value)
    elapsed = (time.time() - start) * 1000  # client sees this latency

    # Replicate in background (simulating network delay)
    def replicate_later():
        time.sleep(delay_ms / 1000)
        for node_name in replicas[1:]:
            cache.nodes[node_name].set(key, value)

    threading.Thread(target=replicate_later, daemon=True).start()
    return elapsed


# Compare write latency
print("⏱️  Write Latency Comparison")
print("=" * 50)

sync_times = []
async_times = []

for i in range(100):
    sync_times.append(sync_write(cache, f"bench:sync:{i}", f"value-{i}"))
    async_times.append(async_write(cache, f"bench:async:{i}", f"value-{i}", delay_ms=50))

# Medians, not means: background replication threads occasionally stall one
# sample, and a single 20 ms outlier would swamp a 0.3 ms average.
sync_median = statistics.median(sync_times)
async_median = statistics.median(async_times)

print(f"  Synchronous:  median {sync_median:.3f} ms per write (avg {sum(sync_times)/len(sync_times):.3f} ms)")
print(f"  Asynchronous: median {async_median:.3f} ms per write (avg {sum(async_times)/len(async_times):.3f} ms)")
print()

# Guard rail: with replication_factor=2, a sync write pays for two round trips
# and an async write pays for one. If that stops being visible, this comparison
# is measuring noise rather than replication.
assert cache.replication_factor == 2
assert sync_median > async_median, (
    f"sync writes ({sync_median:.3f} ms) should cost more than async "
    f"({async_median:.3f} ms) — sync does 2 round trips, async does 1"
)

print(f"  Sync/async ratio: {sync_median/async_median:.2f}× (replication factor is {cache.replication_factor})")
print("  Sync writes are slower because they wait for ALL replicas.")
print("  Async writes return as soon as the primary confirms.")

In [ ]:
# The coherence problem: async replication creates a window of inconsistency

# Flush first
for client in NODES.values():
    client.flushall()

key = "product:price"
OLD_PRICE, NEW_PRICE = "$129.99", "$99.99"

# Seed the OLD value on EVERY replica first. This step matters more than it looks:
# if we skipped it, the lagging replica would simply be empty, and an empty replica
# is just a cache MISS — annoying, but safe, because the reader falls through to
# the database. The dangerous case is a replica that confidently serves the
# PREVIOUS value as if it were current. That's what we want to reproduce.
sync_write(cache, key, OLD_PRICE)
print(f"📖 Before the price change — all replicas agree on {OLD_PRICE}:")
for node, val in cache.get_from_any(key).items():
    print(f"  {node}: {val!r:>12}")

# Now change the price, replicating asynchronously with a 500ms lag
print(f"\n✍️  Price changed to {NEW_PRICE} (async replication, 500ms lag)")
async_write(cache, key, NEW_PRICE, delay_ms=500)

# Immediately read from all replicas
print()
print("📖 Reading from all replicas IMMEDIATELY after async write:")
values = cache.get_from_any(key)
for node, val in values.items():
    status = "✅" if val == NEW_PRICE else "❌ STALE"
    print(f"  {node}: {val!r:>12}  {status}")

# Guard rail: the lesson only lands if a replica really is behind — and behind
# with the OLD PRICE, not with None.
lagging = [n for n, v in values.items() if v != NEW_PRICE]
assert lagging, "async replication produced no staleness window — nothing to teach here"
assert all(values[n] == OLD_PRICE for n in lagging), (
    f"lagging replicas should still serve the old price, got "
    f"{ {n: values[n] for n in lagging} }"
)
print()
print(f"  ⚠️  {len(lagging)} replica(s) are serving {OLD_PRICE} — a real, wrong answer,")
print("     not a miss. A reader routed there bills the customer the old price.")

print()
print("⏳ Waiting 1 second for replication to complete...")
time.sleep(1)

print()
print("📖 Reading from all replicas AFTER replication:")
values = cache.get_from_any(key)
for node, val in values.items():
    status = "✅" if val == NEW_PRICE else "❌ STALE"
    print(f"  {node}: {val!r:>12}  {status}")

# ...and it must actually converge.
assert all(v == NEW_PRICE for v in values.values()), (
    f"replicas never converged after 1s: {values}"
)

print()
print("🔑 Key insight: async replication is eventually consistent.")
print("   There's a brief window where replicas disagree.")
print("   For caches, this is usually acceptable — stale data for a few ms")
print("   is better than slow writes or unavailability.")

---

## 🗑️ Part 3: Cache Invalidation Strategies

When the source of truth (database) changes, the cache becomes **stale**.
We need a strategy to remove or update the stale data.

There are three main approaches:

| Strategy | How It Works | Pros | Cons |
|----------|-------------|------|------|
| **TTL-based** | Keys auto-expire after N seconds | Simple, no coordination | Stale for up to TTL |
| **Write-invalidate** | Delete cached key on write | Simple, always fresh on next read | Cache miss after every write |
| **Write-update** | Update cached value on write | Always fresh, no miss | More complex, wasted if rarely read |

In [ ]:
# Strategy 1: TTL-based Invalidation
# Keys automatically expire after a set time

for client in NODES.values():
    client.flushall()

node = NODES["node-1"]

# Set a key with a 3-second TTL
node.set("product:42:price", "$99.99", ex=3)  # ex = expire in N seconds

print("⏰ TTL-based Invalidation Demo")
print("=" * 50)

for i in range(5):
    value = node.get("product:42:price")
    ttl = node.ttl("product:42:price")
    status = "✅ cached" if value else "❌ expired"
    print(f"  t={i}s: value={value!r:<12} TTL={ttl}s  {status}")
    time.sleep(1)

print()
print("💡 After 3 seconds, the key expires automatically.")
print("   Next read will be a cache miss → fetch from DB → re-cache.")
print()
print("   Trade-off: data can be stale for up to TTL seconds.")
print("   Short TTL = fresher data but more DB hits.")
print("   Long TTL = fewer DB hits but staler data.")

In [ ]:
# Strategy 2: Write-Invalidate
# When data changes in the DB, DELETE the cached key on ALL replicas

for client in NODES.values():
    client.flushall()

def write_invalidate(cache: ReplicatedCache, key: str):
    """Delete a key from all replicas that hold it."""
    replicas = cache._get_replicas(key)
    for node_name in replicas:
        cache.nodes[node_name].delete(key)
    return replicas


# Simulate: write data, then invalidate when DB changes
key = "user:alice:profile"
sync_write(cache, key, json.dumps({"name": "Alice", "email": "alice@old.com"}))

print("1️⃣  Initial state (cached on 2 replicas):")
for node, val in cache.get_from_any(key).items():
    print(f"   {node}: {val}")

# Simulate: Alice changes her email in the database
print("\n2️⃣  Alice changes email in the database...")
print("   DB: {\"name\": \"Alice\", \"email\": \"alice@new.com\"}")

# Invalidate the cached copy
invalidated = write_invalidate(cache, key)
print(f"   Invalidated key on: {invalidated}")

print("\n3️⃣  After invalidation:")
for node, val in cache.get_from_any(key).items():
    status = "(cache miss — will fetch from DB)" if val is None else val
    print(f"   {node}: {status}")

print()
print("✅ Write-invalidate ensures the NEXT read gets fresh data from the DB.")
print("   Downside: that next read is a cache miss (slightly slower).")

In [ ]:
# Strategy 3: Write-Update (push new value to all replicas)

for client in NODES.values():
    client.flushall()

def write_update(cache: ReplicatedCache, key: str, new_value: str):
    """Update a key on all replicas with the new value."""
    replicas = cache._get_replicas(key)
    for node_name in replicas:
        cache.nodes[node_name].set(key, new_value)
    return replicas


key = "user:bob:profile"
sync_write(cache, key, json.dumps({"name": "Bob", "score": 100}))

print("1️⃣  Initial state:")
for node, val in cache.get_from_any(key).items():
    print(f"   {node}: {val}")

# Bob's score changes in the DB
new_data = json.dumps({"name": "Bob", "score": 250})
print(f"\n2️⃣  Bob's score updated in DB to 250...")

# Push the update to all replicas
updated = write_update(cache, key, new_data)
print(f"   Updated cache on: {updated}")

print("\n3️⃣  After write-update:")
for node, val in cache.get_from_any(key).items():
    print(f"   {node}: {val}")

print()
print("✅ Write-update keeps the cache always warm — no cache miss.")
print("   Downside: extra write traffic, wasted if key is rarely read.")

---

## 📡 Part 4: Pub/Sub Invalidation (Cross-Node Coordination)

In a real distributed cache, nodes need to **notify each other** when data changes.
Redis Pub/Sub is a simple way to broadcast invalidation messages.

The pattern:
1. When a key changes, publish an invalidation message to a channel
2. All nodes subscribe to that channel
3. When they receive a message, they delete the local copy

This is how systems like Facebook's Memcached (TAO) coordinate invalidation.

> ⚠️ **What our three-container setup can and can't show.** Redis Pub/Sub does not
> cross server boundaries, and our three nodes are three *independent* Redis
> servers — so `cache:invalidate` is really three separate channels, and the
> publisher below has to fan out to each one. That works for the demo, but it
> means the invalidator must know every node and stay up while it publishes: lose
> the publisher halfway through the fan-out and some nodes keep stale data forever.
> A real deployment publishes once to a **shared** bus (one Redis, Redis Streams,
> or Kafka) that every cache node subscribes to.

In [ ]:
# Pub/Sub Invalidation Demo
# Each Redis server has its OWN copy of the channel (pub/sub does not cross
# server boundaries), so the invalidator fans out to all three.

for client in NODES.values():
    client.flushall()

INVALIDATION_CHANNEL = "cache:invalidate"
invalidation_log = []           # track what happened
_log_lock = threading.Lock()    # listeners append from three different threads
stop_listening = threading.Event()


def start_invalidation_listener(node_name: str, client: redis.Redis) -> threading.Thread:
    """Subscribe to invalidation events and delete local keys."""
    # Use a separate connection for subscribing — a connection in subscribe mode
    # can't run normal commands, so the DELETE has to go over `client`.
    sub_client = redis.Redis(
        host="localhost",
        port=client.connection_pool.connection_kwargs["port"],
        decode_responses=True,
    )
    pubsub = sub_client.pubsub(ignore_subscribe_messages=True)
    pubsub.subscribe(INVALIDATION_CHANNEL)

    def listener():
        # Poll with a timeout instead of the more obvious blocking
        # `for message in pubsub.listen()`. A blocking listen() never returns,
        # so the thread outlives the cell that started it — and when it later
        # raises (or the kernel tears its stdout down) you get the classic
        # "ValueError: I/O operation on closed file" from a cell that already
        # finished. A stop flag + bounded poll keeps the thread inside its cell.
        try:
            while not stop_listening.is_set():
                message = pubsub.get_message(timeout=0.1)
                if not message or message["type"] != "message":
                    continue
                key_to_invalidate = message["data"]
                if client.delete(key_to_invalidate):
                    with _log_lock:
                        invalidation_log.append(
                            f"{node_name}: deleted '{key_to_invalidate}'"
                        )
        finally:
            pubsub.close()
            sub_client.close()

    t = threading.Thread(target=listener, name=f"listener-{node_name}", daemon=True)
    t.start()
    return t


def wait_for_subscribers(timeout: float = 10.0) -> dict:
    """
    Block until every node reports a live subscriber on the channel.

    `pubsub.subscribe()` only *sends* the SUBSCRIBE command; the server registers
    it a moment later. Sleeping a fixed 0.5s and hoping is exactly the kind of
    wall-clock race this repo is full of — so ask Redis directly, with a bound.
    """
    deadline = time.time() + timeout
    while True:
        counts = {
            name: dict(client.pubsub_numsub(INVALIDATION_CHANNEL))[INVALIDATION_CHANNEL]
            for name, client in NODES.items()
        }
        if all(c >= 1 for c in counts.values()):
            return counts
        if time.time() > deadline:
            raise TimeoutError(
                f"listeners never registered within {timeout}s — "
                f"PUBSUB NUMSUB {INVALIDATION_CHANNEL} = {counts}, expected >= 1 on every node"
            )
        time.sleep(0.02)


listeners: list[threading.Thread] = []
try:
    # Start listeners on all nodes, then WAIT until Redis confirms each one.
    for _name, _client in NODES.items():
        listeners.append(start_invalidation_listener(_name, _client))
    subscriber_counts = wait_for_subscribers()
    print(f"👂 Subscribers registered (PUBSUB NUMSUB): {subscriber_counts}")

    # Write the same key to all 3 nodes (simulating full replication)
    key = "config:feature_flags"
    value = json.dumps({"dark_mode": True, "beta_chat": False})

    for client in NODES.values():
        client.set(key, value)

    print("\n1️⃣  Key stored on ALL nodes:")
    for name, client in NODES.items():
        print(f"   {name}: {client.get(key)}")
    assert all(c.get(key) == value for c in NODES.values()), "setup failed: key not on every node"

    # Now broadcast an invalidation message
    print(f"\n2️⃣  Broadcasting invalidation for '{key}'...")
    # Publish to the channel on each node so local subscribers hear it.
    # `publish` returns how many subscribers received the message — 0 would mean
    # the demo below "passes" for the wrong reason, so check it.
    for name, client in NODES.items():
        receivers = client.publish(INVALIDATION_CHANNEL, key)
        assert receivers >= 1, f"nobody was subscribed on {name} — listener thread never started"

    # Wait for the listeners to act (they run in background threads).
    deadline = time.time() + 5
    while time.time() < deadline and any(c.get(key) is not None for c in NODES.values()):
        time.sleep(0.02)

    print("\n3️⃣  After invalidation:")
    for name, client in NODES.items():
        val = client.get(key)
        status = "✅ invalidated" if val is None else f"⚠️ still has: {val}"
        print(f"   {name}: {status}")

    print("\n📋 Invalidation log:")
    with _log_lock:
        for entry in invalidation_log:
            print(f"   {entry}")

    # Guard rail: every node must have dropped its copy, and every node must have
    # said so. A silent partial invalidation is exactly the bug this pattern hides.
    survivors = {n: c.get(key) for n, c in NODES.items() if c.get(key) is not None}
    assert not survivors, f"these nodes still hold the stale key: {survivors}"
    with _log_lock:
        assert len(invalidation_log) == len(NODES), (
            f"expected one delete per node, got {len(invalidation_log)}: {invalidation_log}"
        )
finally:
    # Nothing may outlive this cell: stop the listeners and join them here.
    stop_listening.set()
    for t in listeners:
        t.join(timeout=5)

still_running = [t.name for t in listeners if t.is_alive()]
assert not still_running, f"invalidation listeners outlived their cell: {still_running}"

print()
print("✅ Pub/Sub lets us coordinate invalidation across all nodes in real time.")
print("   (Listeners stopped and joined — no background thread outlives this cell.)")
print("   In production, you'd use a dedicated message bus (Kafka, Redis Streams, etc.)")

---

## 📏 Part 5: Measuring Staleness

How stale is your cache? Let's build a simple tool to measure the "staleness window" —
the time between a write and when all replicas see the new value.

In [ ]:
# Measure replication lag: how long until all replicas see the update?

for client in NODES.values():
    client.flushall()

POLL_INTERVAL_S = 0.01   # 10ms — this is also the resolution of our measurement
CONVERGE_TIMEOUT_S = 5.0


def measure_staleness(cache: ReplicatedCache, key: str, value: str,
                      replication_delay_ms: int) -> dict:
    """Write with async replication and measure how long replicas are stale."""
    replicas = cache._get_replicas(key)
    primary = replicas[0]

    # Write to primary
    write_time = time.time()
    cache.nodes[primary].set(key, value)

    # Replicate after delay
    def replicate():
        time.sleep(replication_delay_ms / 1000)
        for node_name in replicas[1:]:
            cache.nodes[node_name].set(key, value)

    threading.Thread(target=replicate, daemon=True).start()

    # Poll replicas until they all have the new value.
    # Bounded: an unbounded `while True` would hang the notebook forever if the
    # replication thread ever died.
    stale_durations = {}
    deadline = write_time + CONVERGE_TIMEOUT_S
    for node_name in replicas[1:]:
        while True:
            if cache.nodes[node_name].get(key) == value:
                stale_durations[node_name] = (time.time() - write_time) * 1000
                break
            if time.time() > deadline:
                raise TimeoutError(
                    f"{node_name} never converged on {key!r} within {CONVERGE_TIMEOUT_S}s"
                )
            time.sleep(POLL_INTERVAL_S)

    return stale_durations


print("📏 Measuring Staleness Window")
print("=" * 50)

measured = {}
for delay in [50, 100, 200, 500]:
    key = f"stale_test:{delay}"
    durations = measure_staleness(cache, key, f"value-{delay}", delay)
    measured[delay] = max(durations.values())
    for node, ms in durations.items():
        print(f"  Delay={delay:>3}ms → {node} was stale for {ms:.0f}ms")

    # Guard rail: the measured window must actually track the injected lag.
    # Lower bound = the lag itself; upper bound = lag + one poll interval + slack.
    for node, ms in durations.items():
        assert ms >= delay * 0.9, (
            f"{node} converged in {ms:.0f}ms but replication was delayed {delay}ms — "
            "the measurement is not seeing the lag"
        )
        assert ms < delay + 250, (
            f"{node} took {ms:.0f}ms to converge, far beyond the {delay}ms lag "
            f"(+{POLL_INTERVAL_S*1000:.0f}ms poll resolution)"
        )

# ...and a bigger configured lag must produce a bigger observed window.
windows = [measured[d] for d in [50, 100, 200, 500]]
assert all(a < b for a, b in zip(windows, windows[1:])), (
    f"staleness window should grow with replication delay, got {measured}"
)

print()
print("🔑 The staleness window = replication delay + network latency.")
print("   In production, this is typically 1-10ms within a data center.")
print("   For caches, this is usually acceptable — we trade consistency for speed.")

---

## 🧪 Try It Yourself

1. **Change the replication factor** from 2 to 3 — now every key lives on all nodes.
   How does this affect write latency for synchronous replication?

2. **Simulate a node failure**: stop one Redis container (`docker stop dcache-redis-2`)
   and try reading keys. Does the cache still work with replication factor 2?

3. **Combine TTL + Write-Invalidate**: Set keys with a 60s TTL, but also invalidate
   on write. This gives you a safety net — even if invalidation fails, the TTL
   ensures data refreshes eventually.

4. **Open RedisInsight** and watch keys appear/disappear as you run invalidation.

## 📝 Key Takeaways

| Concept | Key Insight |
|---------|-------------|
| Replication | Copies data to survive node failures |
| Sync replication | Strong consistency, slower writes |
| Async replication | Fast writes, brief staleness window |
| TTL invalidation | Simple, but stale for up to TTL seconds |
| Write-invalidate | Delete on write, cache miss on next read |
| Write-update | Push new value, no cache miss |
| Pub/Sub | Real-time cross-node coordination |

## ➡️ Next: Notebook 3 — Consistent Hashing for Cache Routing

We've seen partitioning (Notebook 1) and coherence (this notebook).  
Now let's deep-dive into **consistent hashing** — the algorithm that makes
distributed caches work in production.

In [ ]:
# Cleanup
for client in NODES.values():
    client.flushall()
print("🧹 All nodes flushed.")